In [109]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, f1_score,
                             precision_score, recall_score,
                             average_precision_score,
                             roc_auc_score)
import optuna
import json


In [88]:
SEED = 4266
DATA_DIR = "../../data/processed"
DROP_OUT = 0.3
LEARNING_RATE = 1e-3
HIDDEN_DIM = 100
INPUT_DIM = 59
OUTPUT_DIM = 5
LAYER_DIM = 1

In [89]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [90]:
# feats
X_train = np.load(f"{DATA_DIR}/X_train.npy")
X_val   = np.load(f"{DATA_DIR}/X_val.npy")
X_test  = np.load(f"{DATA_DIR}/X_test.npy")

# src-des
pairs_train = np.load(f"{DATA_DIR}/pairs_train.npy", allow_pickle=True)
pairs_val   = np.load(f"{DATA_DIR}/pairs_val.npy", allow_pickle=True)
pairs_test  = np.load(f"{DATA_DIR}/pairs_test.npy", allow_pickle=True)

# labels
y_train = np.load(f"{DATA_DIR}/y_train.npy")
y_val   = np.load(f"{DATA_DIR}/y_val.npy")
y_test  = np.load(f"{DATA_DIR}/y_test.npy")


In [91]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)

In [92]:
pos_counts = y_train.sum(dim=0)
neg_counts = len(y_train) - pos_counts
pos_weight = (neg_counts / pos_counts.clamp(min=1))

In [93]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn

In [94]:
model = LSTMModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, layer_dim=LAYER_DIM, output_dim=OUTPUT_DIM, dropout=DROP_OUT)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [95]:
# num_epochs = 10
# h0, c0 = None, None

# for epoch in range(num_epochs):
#     model.train()
#     optimizer.zero_grad()

#     outputs, h0, c0 = model(X_train, h0, c0)
# # Source - https://stackoverflow.com/a/73161912
# # Posted by ki-ljl
# # Retrieved 2026-10-08, License - CC BY-SA 4.0

# def get_best_parameters(args, Dtr, Val):
#     def objective(trial):
#         model = TransformerModel(args).to(args.device)
#         loss_function = nn.MSELoss().to(args.device)
#         optimizer = trial.suggest_categorical('optimizer',
#                                               [torch.optim.SGD,
#                                                torch.optim.RMSprop,
#                                                torch.optim.Adam])(
#             model.parameters(), lr=trial.suggest_loguniform('lr', 5e-4, 1e-2))
#         print('training...')
#         epochs = 10
#         val_loss = 0
#         for epoch in range(epochs):
#             train_loss = []
#             for batch_idx, (seq, target) in enumerate(Dtr, 0):
#                 seq, target = seq.to(args.device), target.to(args.device)
#                 optimizer.zero_grad()
#                 y_pred = model(seq)
#                 loss = loss_function(y_pred, target)
#                 train_loss.append(loss.item())
#                 loss.backward()
#                 optimizer.step()
#             # validation
#             val_loss = get_val_loss(args, model, Val)

#             print('epoch {:03d} train_loss {:.8f} val_loss {:.8f}'.format(epoch, np.mean(train_loss), val_loss))
#             model.train()

#         return val_loss

#     sampler = optuna.samplers.TPESampler()
#     study = optuna.create_study(sampler=sampler, direction='minimize')
#     study.optimize(func=objective, n_trials=5)
#     pruned_trials = study.get_trials(deepcopy=False,
#                                      states=tuple([TrialState.PRUNED]))
#     complete_trials = study.get_trials(deepcopy=False,
#                                        states=tuple([TrialState.COMPLETE]))
#     best_trial = study.best_trial
#     print('val_loss = ', best_trial.value)
#     for key, value in best_trial.params.items():
#         print("{}: {}".format(key, value))

#     loss = criterion(outputs, y_train)
#     loss.backward()
#     optimizer.step()

#     h0, c0 = h0.detach(), c0.detach()

#     if (epoch+1) % 1 == 0:
#         print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')




In [96]:
model.eval()
with torch.no_grad():
    logits, _, _ = model(X_test)          # (N, C) raw scores
    probs = torch.sigmoid(logits)         # (N, C) independent probs in [0,1]
    preds = (probs > 0.5).float()         # (N, C) multi-hot

y_true = y_test.cpu().numpy()             # (N, C) multi-hot float
y_prob = probs.cpu().numpy()              # (N, C) probabilities
y_pred = preds.cpu().numpy()              # (N, C) binary predictions

print("logits.shape:", logits.shape)
print("probs.shape: ", probs.shape)
print("preds.shape: ", preds.shape)
print("y_true.shape:", y_true.shape)
print("y_prob.shape:", y_prob.shape)
print("y_pred.shape:", y_pred.shape)

print("\nSample probabilities:", y_prob[0])
print("Sample predictions:  ", y_pred[0])
print("Sample ground truth: ", y_true[0])
print("\nPositive rate per class:", y_pred.mean(axis=0))

logits.shape: torch.Size([47364, 5])
probs.shape:  torch.Size([47364, 5])
preds.shape:  torch.Size([47364, 5])
y_true.shape: (47364, 5)
y_prob.shape: (47364, 5)
y_pred.shape: (47364, 5)

Sample probabilities: [0.4786024  0.4965815  0.50243616 0.52272326 0.5024117 ]
Sample predictions:   [0. 0. 1. 1. 1.]
Sample ground truth:  [0. 0. 0. 0. 1.]

Positive rate per class: [0.02328773 0.00496157 0.98266613 0.9971497  0.93906766]


In [97]:
print("true positives per class: ", y_true.sum(axis=0))
print("true positive rate:        ", y_true.mean(axis=0))
print("pred positive rate:        ", y_pred.mean(axis=0))

true positives per class:  [3.9000e+01 5.1000e+01 9.1000e+01 3.1000e+01 4.7152e+04]
true positive rate:         [8.2341017e-04 1.0767672e-03 1.9212904e-03 6.5450551e-04 9.9552405e-01]
pred positive rate:         [0.02328773 0.00496157 0.98266613 0.9971497  0.93906766]


In [98]:
# Source - https://stackoverflow.com/a/73161912
# Posted by ki-ljl
# Retrieved 2026-10-08, License - CC BY-SA 4.0

def get_best_parameters(args, Dtr, Val):
    def objective(trial):
        model = TransformerModel(args).to(args.device)
        loss_function = nn.MSELoss().to(args.device)
        optimizer = trial.suggest_categorical('optimizer',
                                              [torch.optim.SGD,
                                               torch.optim.RMSprop,
                                               torch.optim.Adam])(
            model.parameters(), lr=trial.suggest_loguniform('lr', 5e-4, 1e-2))
        print('training...')
        epochs = 10
        val_loss = 0
        for epoch in range(epochs):
            train_loss = []
            for batch_idx, (seq, target) in enumerate(Dtr, 0):
                seq, target = seq.to(args.device), target.to(args.device)
                optimizer.zero_grad()
                y_pred = model(seq)
                loss = loss_function(y_pred, target)
                train_loss.append(loss.item())
                loss.backward()
                optimizer.step()
            # validation
            val_loss = get_val_loss(args, model, Val)

            print('epoch {:03d} train_loss {:.8f} val_loss {:.8f}'.format(epoch, np.mean(train_loss), val_loss))
            model.train()

        return val_loss

    sampler = optuna.samplers.TPESampler()
    study = optuna.create_study(sampler=sampler, direction='minimize')
    study.optimize(func=objective, n_trials=5)
    pruned_trials = study.get_trials(deepcopy=False,
                                     states=tuple([TrialState.PRUNED]))
    complete_trials = study.get_trials(deepcopy=False,
                                       states=tuple([TrialState.COMPLETE]))
    best_trial = study.best_trial
    print('val_loss = ', best_trial.value)
    for key, value in best_trial.params.items():
        print("{}: {}".format(key, value))


In [99]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim,
                            batch_first=True,
                            dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn    = nn.Linear(hidden_dim, 1)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)                       # (B, T, H) — zeros state by default
        # attention pooling over time
        scores  = self.attn(out).squeeze(-1)         # (B, T)
        weights = torch.softmax(scores, dim=1)       # (B, T)
        pooled  = (out * weights.unsqueeze(-1)).sum(1)  # (B, H)
        pooled  = self.dropout(pooled)
        return self.fc(pooled)

In [100]:
def train_eval(hidden_dim, layer_dim, dropout, lr, weight_decay,
               optimizer_name, batch_size, epochs=20, patience=4,
               device=device, seed=SEED):

    torch.manual_seed(seed)
    np.random.seed(seed)


    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, drop_last=False)

    
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    OptimCls = {"adam": torch.optim.Adam,
                "adamw": torch.optim.AdamW,
                "rmsprop": torch.optim.RMSprop}[optimizer_name]
    optimizer = OptimCls(model.parameters(), lr=lr, weight_decay=weight_decay)

    X_val_dev = X_val.to(device)
    y_val_np  = y_val.numpy()

    best_auc, best_state, no_improve = -1.0, None, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_dev)
            val_probs  = torch.sigmoid(val_logits).cpu().numpy()

        aucs = []
        for c in range(y_val_np.shape[1]):
            col = y_val_np[:, c]
            if 0 < col.sum() < len(col):
                aucs.append(roc_auc_score(col, val_probs[:, c]))
        val_auc = float(np.mean(aucs)) if aucs else 0.0

        if val_auc > best_auc + 1e-4:
            best_auc = val_auc
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_auc, model

In [101]:
def objective(trial):
    hidden_dim    = trial.suggest_categorical("hidden_dim", [32, 64, 100, 128, 256])
    layer_dim     = trial.suggest_int("layer_dim", 1, 2)
    dropout       = trial.suggest_float("dropout", 0.0, 0.5, step=0.1)
    lr            = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    optimizer_name= trial.suggest_categorical("optimizer", ["adam", "adamw", "rmsprop"])
    batch_size    = trial.suggest_categorical("batch_size", [128, 256, 512])

    auc, _ = train_eval(hidden_dim, layer_dim, dropout, lr, weight_decay,
                        optimizer_name, batch_size)
    trial.set_user_attr("val_auc", auc)
    return auc

In [102]:
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="maximize", sampler=sampler,
                            study_name="lstm_multilabel")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best val AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-10-09 00:14:51,378] A new study created in memory with name: lstm_multilabel


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-09 00:15:59,523] Trial 0 finished with value: 0.9999125049463429 and parameters: {'hidden_dim': 64, 'layer_dim': 1, 'dropout': 0.4, 'lr': 0.004847273740403608, 'weight_decay': 1.316393745931565e-06, 'optimizer': 'adam', 'batch_size': 512}. Best is trial 0 with value: 0.9999125049463429.
[I 2026-10-09 00:17:01,564] Trial 1 finished with value: 0.9999537900120237 and parameters: {'hidden_dim': 100, 'layer_dim': 1, 'dropout': 0.0, 'lr': 0.0004160275946310532, 'weight_decay': 0.0001289396821735606, 'optimizer': 'adamw', 'batch_size': 128}. Best is trial 1 with value: 0.9999537900120237.
[I 2026-10-09 00:20:26,393] Trial 2 finished with value: 0.9999156329750638 and parameters: {'hidden_dim': 100, 'layer_dim': 2, 'dropout': 0.5, 'lr': 0.0005919451036137254, 'weight_decay': 3.0940459482726795e-05, 'optimizer': 'rmsprop', 'batch_size': 256}. Best is trial 1 with value: 0.9999537900120237.
[I 2026-10-09 00:23:58,266] Trial 3 finished with value: 0.9998303361196502 and parameters: {'

In [107]:
print("Best val AUC:", study.best_value)
print("Best params :", study.best_params)

Best val AUC: 0.9999860713166635
Best params : {'hidden_dim': 64, 'layer_dim': 2, 'dropout': 0.4, 'lr': 0.0002563908942916025, 'weight_decay': 1.791262473675998e-06, 'optimizer': 'adamw', 'batch_size': 256}


In [108]:
with open("best_params.json", "w") as f:
    json.dump(best, f, indent=2)

In [111]:
CLASS_NAMES = ["Backdoor", "CommInj", "DoS", "Reconn", "normal"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best = study.best_params
print("Best params:", best)

model = LSTMModel(INPUT_DIM, best["hidden_dim"], best["layer_dim"],
                  OUTPUT_DIM, best["dropout"]).to(device)

pos_counts = y_train.sum(0).clamp(min=1)
neg_counts = len(y_train) - pos_counts
pos_weight = (neg_counts / pos_counts).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
OptimCls = {"adam": torch.optim.Adam, "adamw": torch.optim.AdamW,
            "rmsprop": torch.optim.RMSprop}[best["optimizer"]]
optimizer = OptimCls(model.parameters(),
                     lr=best["lr"], weight_decay=best["weight_decay"])

loader = DataLoader(TensorDataset(X_train, y_train),
                    batch_size=best["batch_size"], shuffle=True)
for epoch in range(30):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1:2d}  loss={total/len(loader.dataset):.4f}")

torch.save(model.state_dict(), "lstm_best.pt")


Best params: {'hidden_dim': 64, 'layer_dim': 2, 'dropout': 0.4, 'lr': 0.0002563908942916025, 'weight_decay': 1.791262473675998e-06, 'optimizer': 'adamw', 'batch_size': 256}
epoch  1  loss=0.6755
epoch  2  loss=0.1865
epoch  3  loss=0.1018
epoch  4  loss=0.0752
epoch  5  loss=0.0537
epoch  6  loss=0.0420
epoch  7  loss=0.0497
epoch  8  loss=0.0478
epoch  9  loss=0.0318
epoch 10  loss=0.0247
epoch 11  loss=0.0352
epoch 12  loss=0.0290
epoch 13  loss=0.0295
epoch 14  loss=0.0247
epoch 15  loss=0.0252
epoch 16  loss=0.0247
epoch 17  loss=0.0180
epoch 18  loss=0.0237
epoch 19  loss=0.0166
epoch 20  loss=0.0199
epoch 21  loss=0.0163
epoch 22  loss=0.0106
epoch 23  loss=0.0182
epoch 24  loss=0.0137
epoch 25  loss=0.0099
epoch 26  loss=0.0133
epoch 27  loss=0.0162
epoch 28  loss=0.0186
epoch 29  loss=0.0119
epoch 30  loss=0.0100


In [113]:
model.eval()
test_logits_list = []
with torch.no_grad():
    for i in range(0, len(X_test), 512):
        test_logits_list.append(model(X_test[i:i+512].to(device)).cpu())
test_logits = torch.cat(test_logits_list)

test_loss = torch.nn.functional.binary_cross_entropy_with_logits(
    test_logits.to(device), y_test.to(device),
    pos_weight=pos_weight).item()

test_probs = torch.sigmoid(test_logits).numpy()
y_true_np  = y_test.numpy()
y_pred_np  = (test_probs > 0.5).astype(int)

f1   = f1_score(y_true_np, y_pred_np, average="macro", zero_division=0)
prec = precision_score(y_true_np, y_pred_np, average="macro", zero_division=0)
rec  = recall_score(y_true_np, y_pred_np, average="macro", zero_division=0)
pr_auc = average_precision_score(y_true_np, test_probs, average="macro")

print(f"\nTest Loss: {test_loss:.4f} | F1: {f1:.4f} | "
      f"Precision: {prec:.4f} | Recall: {rec:.4f} | PR-AUC: {pr_auc:.4f}\n")
print("Classification Report (multilabel, threshold=0.5):")
print(classification_report(y_true_np, y_pred_np,
                            target_names=CLASS_NAMES,
                            digits=4, zero_division=0))

print(f"\n{'class':>8} {'pos':>6} {'base':>9} {'AUC':>7} {'AP':>7} {'AP/base':>9}")
for c, name in enumerate(CLASS_NAMES):
    col = y_true_np[:, c]
    if 0 < col.sum() < len(col):
        auc = roc_auc_score(col, test_probs[:, c])
        ap  = average_precision_score(col, test_probs[:, c])
        base = col.mean()
        print(f"{name:>8} {int(col.sum()):>6} {base:>9.5f} "
              f"{auc:>7.4f} {ap:>7.4f} {ap/base if base else 0:>9.2f}x")


Test Loss: 0.0778 | F1: 0.7937 | Precision: 0.7203 | Recall: 0.9766 | PR-AUC: 0.9624

Classification Report (multilabel, threshold=0.5):
              precision    recall  f1-score   support

    Backdoor     0.2534    0.9487    0.4000        39
     CommInj     0.5426    1.0000    0.7034        51
         DoS     1.0000    1.0000    1.0000        91
      Reconn     0.8056    0.9355    0.8657        31
      normal     1.0000    0.9989    0.9994     47152

   micro avg     0.9966    0.9988    0.9977     47364
   macro avg     0.7203    0.9766    0.7937     47364
weighted avg     0.9987    0.9988    0.9985     47364
 samples avg     0.9979    0.9988    0.9982     47364


   class    pos      base     AUC      AP   AP/base
Backdoor     39   0.00082  0.9997  0.8606   1045.22x
 CommInj     51   0.00108  1.0000  0.9849    914.69x
     DoS     91   0.00192  1.0000  1.0000    520.48x
  Reconn     31   0.00065  1.0000  0.9667   1476.93x
  normal  47152   0.99552  0.9999  1.0000      1.00x
